In [1]:
import os
import shutil
import time
from glob import glob
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm

In [2]:
def change_file_time(file, t):
    timestamp_str = str(t)
    parsed_time = time.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
    ts = time.mktime(parsed_time)
    os.utime(file, (ts, ts))

In [3]:
df = pd.read_csv('wildlife-insights_4fc92e40-94fe-48d3-a51f-5ac1a03f5e03_project-2007160_data/images_2007160.csv')
len(df)

17368

In [4]:
len(df['deployment_id'].unique())

32

In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [7]:
df = df[~df['common_name'].isin(['Human-Camera Trapper', 'Human-Faces', 'Human', 'Blank'])]
len(df)

8774

In [8]:
df = df.sort_values(by='timestamp')

In [9]:
df.reset_index(drop=True, inplace=True)

In [10]:
df['deployment_id'] = df['deployment_id'].str.replace('/', '-')

In [16]:
df['location'].to_csv('animal_images.csv', index=False, header=False)

In [14]:
Path('animal_images').mkdir(exist_ok=True)

```
cat images_animals_cam2 02-10-2024.csv | gsutil -m cp -I detection_frames
```

In [17]:
df.head()

,project_id,deployment_id,image_id,filename,sequence_id,location,wi_taxon_id,class,order,family,genus,species,common_name,timestamp,bounding_boxes
0,2007160,PM16 02-10-2024,e4c20077-458c-43a5-9f36-599fe29c25ce,RCNX0088.JPG,8661725,gs://145625598251_2007160_901_pilot_24__main/d...,5c7ce479-8a45-40b3-ae21-7c97dfae22f5,Mammalia,Cetartiodactyla,Cervidae,Odocoileus,virginianus,White-tailed Deer,2024-02-10 13:35:09,"{""{\""detectionBox\"":[0.404988974,0.278540134,0..."
1,2007160,PM16 02-10-2024,fca22548-ea22-4752-beff-910f53c65318,RCNX0089.JPG,8661725,gs://145625598251_2007160_901_pilot_24__main/d...,5c7ce479-8a45-40b3-ae21-7c97dfae22f5,Mammalia,Cetartiodactyla,Cervidae,Odocoileus,virginianus,White-tailed Deer,2024-02-10 13:35:10,"{""{\""detectionBox\"":[0.495519221,0.444105357,0..."
2,2007160,PM16 02-10-2024,8dcf1372-1501-4636-850e-f69018f11d29,RCNX0090.JPG,8661725,gs://145625598251_2007160_901_pilot_24__main/d...,5c7ce479-8a45-40b3-ae21-7c97dfae22f5,Mammalia,Cetartiodactyla,Cervidae,Odocoileus,virginianus,White-tailed Deer,2024-02-10 13:35:11,"{""{\""detectionBox\"":[0.599095643,0.407299489,0..."
3,2007160,PM16 02-10-2024,d94dc2da-0a4e-497b-9926-88a8276ecb67,RCNX0091.JPG,8661725,gs://145625598251_2007160_901_pilot_24__main/d...,5c7ce479-8a45-40b3-ae21-7c97dfae22f5,Mammalia,Cetartiodactyla,Cervidae,Odocoileus,virginianus,White-tailed Deer,2024-02-10 13:35:32,"{""{\""detectionBox\"":[0.400481373,0,0.978527367..."
4,2007160,PM16 02-10-2024,aa01bf3b-4365-471d-86c7-16496348e76d,RCNX0092.JPG,8661725,gs://145625598251_2007160_901_pilot_24__main/d...,5c7ce479-8a45-40b3-ae21-7c97dfae22f5,Mammalia,Cetartiodactyla,Cervidae,Odocoileus,virginianus,White-tailed Deer,2024-02-10 13:35:33,"{""{\""detectionBox\"":[0.503982782,0.2465882,0.9..."


In [18]:
for x in df['deployment_id'].unique():
    Path(f'animal_images/{x}').mkdir(exist_ok=True, parents=True)

In [20]:
for x, y, z, t in zip(df['location'], df['deployment_id'], df['image_id'], df['timestamp']):
    out_path = f'animal_images/{y}/{z}{Path(x).suffix}'
    shutil.move(f'animal_images/{Path(x).name}', out_path)
    change_file_time(out_path, t)